# NutriMatch Paper Figures With Diet Data Enhancement Arms

This notebook recreates the paper-style Figure 3 analyses for TRE/HPP, then adds the Diet Data Enhancement feature sets.

Paper-style comparisons:

- Age + sex
- Age + sex + paper-basic nutrients
- Age + sex + NutriMatch all nutrients
- Age + sex + enhanced Diet Data Enhancement feature sets

Outputs are written under `downstream_analysis/tasks/nutrimatch_paper_figures_with_enhancements/outputs/` so long model training can run from the console and this notebook can later load the saved results for tables and plots.

In [ ]:
from pathlib import Path
import gc
import json
import os
import re
import sys
import warnings

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import pearsonr
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier, HistGradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    f1_score,
    mean_squared_error,
    r2_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import KFold, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

try:
    from IPython.display import display
except Exception:
    def display(x):
        print(x)

try:
    from lightgbm import LGBMClassifier, LGBMRegressor
    LIGHTGBM_AVAILABLE = True
except Exception:
    LGBMClassifier = None
    LGBMRegressor = None
    LIGHTGBM_AVAILABLE = False

warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)

RANDOM_STATE = int(os.environ.get('DDE_RANDOM_STATE', '42'))
N_SPLITS = int(os.environ.get('DDE_N_SPLITS', '5'))
MIN_N_PER_TARGET = int(os.environ.get('DDE_MIN_N_PER_TARGET', '50'))
MAX_TARGETS = int(os.environ.get('DDE_MAX_PAPER_FIGURE_TARGETS', '36'))
RUN_TRAINING = os.environ.get('DDE_RUN_TRAINING', '0') == '1'
ENABLE_SEX_STRATA = os.environ.get('DDE_ENABLE_SEX_STRATA', '1') == '1'
MODEL_NAME = os.environ.get('DDE_MODEL', 'hist_gradient_boosting')

print('RUN_TRAINING:', RUN_TRAINING)
print('MODEL_NAME:', MODEL_NAME)
print('LightGBM available:', LIGHTGBM_AVAILABLE)

## Paper Anchors

The NutriMatch paper reports Figure 3 as: prediction across phenotypic traits stratified by gender, correlations between Nightingale biomarkers and relative nutrient consumption, and 2-year overweight/obesity prediction with AUROC. The text says HPP nutrients were expanded from 21 to 151, with improvement for body-fat indices, waist circumference, CGM traits, blood biomarkers, and 2-year overweight/obesity prediction.

In [ ]:
PROJECT_ROOT = Path.cwd()
tre_root = Path('/home/ec2-user/studies/Diet_Data_Enhancement_Project/Diet_Data_Enhancement_TRE')
if not (PROJECT_ROOT / 'downstream_analysis').exists() and (tre_root / 'downstream_analysis').exists():
    PROJECT_ROOT = tre_root
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

NOTEBOOK_STEM = 'nutrimatch_paper_figures_with_enhancements'
TASK_DIR = PROJECT_ROOT / 'downstream_analysis/tasks' / NOTEBOOK_STEM
OUT_DIR = TASK_DIR / 'outputs'
FIG_DIR = OUT_DIR / 'figures'
CACHE_DIR = OUT_DIR / 'cache'
LOG_DIR = OUT_DIR / 'logs'
TRE_INPUTS = PROJECT_ROOT / 'tre_inputs'
CVD_OUTPUTS = PROJECT_ROOT / 'downstream_analysis/tasks/cvd/outputs'
for d in [TASK_DIR, OUT_DIR, FIG_DIR, CACHE_DIR, LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

FEATURE_SETS = [
    {'name': 'basic_nutrimatch', 'label': 'NutriMatch all nutrients', 'path': 'outputs/enhanced_hpp/2.nutrimatch_based/hpp_feature_matrix_per_100g.csv', 'feature_mode': 'enriched'},
    {'name': 'denovo_microbiome', 'label': 'De novo microbiome-oriented', 'path': 'outputs/downstream_features/denovo/microbiome/hpp_downstream_feature_table.csv', 'feature_mode': 'kg'},
    {'name': 'denovo_cardiometabolic', 'label': 'De novo cardiometabolic', 'path': 'outputs/downstream_features/denovo/cardiometabolic/hpp_downstream_feature_table.csv', 'feature_mode': 'kg'},
    {'name': 'nutrimatch_microbiome', 'label': 'NutriMatch microbiome-oriented', 'path': 'outputs/downstream_features/nutrimatch_based/microbiome/hpp_downstream_feature_table.csv', 'feature_mode': 'kg'},
    {'name': 'nutrimatch_cardiometabolic', 'label': 'NutriMatch cardiometabolic', 'path': 'outputs/downstream_features/nutrimatch_based/cardiometabolic/hpp_downstream_feature_table.csv', 'feature_mode': 'kg'},
    {'name': 'nutrimatch_broad_diet_health', 'label': 'NutriMatch broad diet-health', 'path': 'outputs/downstream_features/nutrimatch_based/broad_diet_health/hpp_downstream_feature_table.csv', 'feature_mode': 'kg'},
    {'name': 'nutrimatch_mental_health', 'label': 'NutriMatch mental-health', 'path': 'outputs/downstream_features/nutrimatch_based/mental_health/hpp_downstream_feature_table.csv', 'feature_mode': 'kg'},
    {'name': 'denovo_broad_diet_health', 'label': 'De novo broad diet-health', 'path': 'outputs/downstream_features/denovo/broad_diet_health/hpp_downstream_feature_table.csv', 'feature_mode': 'kg'},
    {'name': 'denovo_mental_health', 'label': 'De novo mental-health', 'path': 'outputs/downstream_features/denovo/mental_health/hpp_downstream_feature_table.csv', 'feature_mode': 'kg'},
]
FEATURE_SETS = [fs for fs in FEATURE_SETS if (PROJECT_ROOT / fs['path']).exists() or (CVD_OUTPUTS / fs['name'] / f"X_{fs['name']}_participant.csv").exists()]

PAPER_ARMS = ['age_sex_only', 'paper_basic_nutrients', 'nutrimatch_all']
ARM_ORDER = PAPER_ARMS + [fs['name'] for fs in FEATURE_SETS if fs['name'] != 'basic_nutrimatch']
ARM_LABELS = {
    'age_sex_only': 'Age + sex',
    'paper_basic_nutrients': 'Age + sex + basic nutrients',
    'nutrimatch_all': 'Age + sex + NutriMatch all nutrients',
}
for fs in FEATURE_SETS:
    if fs['name'] != 'basic_nutrimatch':
        ARM_LABELS[fs['name']] = 'Age + sex + ' + fs['label']

print('Project root:', PROJECT_ROOT)
print('Output directory:', OUT_DIR)
print('Feature sets found:', [fs['name'] for fs in FEATURE_SETS])

## Console Runner Status

Long training should be started with the background runner. This cell only checks recent logs and saved files; it does not launch training.

In [ ]:
def newest_tmp_log(prefix='nutrimatch_paper_figures_with_enhancements'):
    logs = sorted(Path('/tmp').glob(prefix + '*.log'), key=lambda p: p.stat().st_mtime, reverse=True)
    return logs[0] if logs else None

latest_pid = LOG_DIR / 'background_training_latest.pid'
print('PID file:', latest_pid, 'exists=', latest_pid.exists())
if latest_pid.exists():
    print('PID:', latest_pid.read_text().strip())
log = newest_tmp_log()
print('Newest /tmp log:', log)
if log and log.exists():
    print(log.read_text(errors='replace')[-4000:])
else:
    print('No temporary log found yet.')

## Loading Helpers

In [ ]:
from downstream_analysis.data_handelling.pheno_loader_export import (
    make_loader,
    load_table_from_loader,
    dataframe_with_index_columns,
)


def read_any(path):
    path = Path(path)
    if path.suffix.lower() == '.parquet':
        return pd.read_parquet(path)
    return pd.read_csv(path, low_memory=False)


def clean_feature_name(name):
    text = str(name).lower()
    text = re.sub(r'^(enriched_|kg_|food_card_)', '', text)
    return re.sub(r'[^a-z0-9]+', '_', text).strip('_')


def normalize_pid_series(s):
    return s.astype(str)


def find_participant_col(df):
    for col in ['participant_id', 'Participant_Study_ID', 'research_stage_id', 'user_id', 'RegistrationCode']:
        if col in df.columns:
            return col
    for col in df.columns:
        text = str(col).lower()
        if 'participant' in text or 'research_stage' in text:
            return col
    return None


def try_load_pheno_table(dataset, table=None):
    try:
        loader = make_loader(dataset, age_sex_dataset=None, errors='warn')
        table_name = table or dataset
        try:
            df = load_table_from_loader(loader, dataset, table_name, required=False)
        except Exception:
            df = None
        if df is None:
            dfs = getattr(loader, 'dfs', {})
            if table_name in dfs:
                df = dataframe_with_index_columns(dfs[table_name])
            elif len(dfs) == 1:
                df = dataframe_with_index_columns(next(iter(dfs.values())))
        return df, loader
    except Exception as exc:
        print(f'Could not load {dataset}/{table or dataset}: {exc}')
        return None, None


def dataframe_brief(dataset, table, df):
    if df is None or df.empty:
        return None
    return {
        'dataset': dataset,
        'table': table,
        'rows': int(len(df)),
        'columns': int(len(df.columns)),
        'participant_col': find_participant_col(df),
        'numeric_columns': int(len(df.select_dtypes(include=np.number).columns)),
    }

## Build Or Reuse Participant-Level Diet Features

This reuses cached participant-level feature matrices from the CVD task when they exist. If a matrix is missing, it builds it from the slim diet table and feature reference table.

In [ ]:
PAPER_BASIC_NUTRIENT_NAMES = [
    'Energy', 'Protein', 'Total lipid (fat)', 'Carbohydrate, by difference',
    'Fiber, total dietary', 'Sodium, Na', 'Water', 'Alcohol, ethyl',
]
BASIC_NUTRIENT_KEYS = {clean_feature_name(x) for x in PAPER_BASIC_NUTRIENT_NAMES}


def choose_ref_food_col(ref):
    for col in ['hpp_food_id', 'food_id']:
        if col in ref.columns:
            return col
    raise ValueError('Could not find hpp_food_id or food_id in feature table')


def feature_columns(ref, ref_food_col, feature_mode):
    if feature_mode == 'embedding':
        cols = [c for c in ref.columns if str(c).startswith('embedding_')]
    else:
        exclude = {ref_food_col, 'food_id', 'hpp_food_id', 'food_name', 'short_food_name', 'product_name'}
        cols = [c for c in ref.columns if c not in exclude and pd.api.types.is_numeric_dtype(ref[c])]
    if not cols:
        numeric = ref.select_dtypes(include=[np.number]).columns.tolist()
        cols = [c for c in numeric if c != ref_food_col]
    return list(dict.fromkeys(cols))


def ensure_diet_slim():
    diet_slim_csv = TRE_INPUTS / 'diet_participant_food_slim.csv'
    diet_events_csv = TRE_INPUTS / 'diet_logging_events.csv'
    if diet_slim_csv.exists():
        print('Using slim diet table:', diet_slim_csv)
        return diet_slim_csv
    if not diet_events_csv.exists():
        print('diet_logging_events.csv not found; loading diet_logging via PhenoLoader.')
        diet, _ = try_load_pheno_table('diet_logging', 'diet_logging_events')
        if diet is None:
            raise FileNotFoundError('Could not load diet_logging_events from TRE.')
        TRE_INPUTS.mkdir(parents=True, exist_ok=True)
        diet.to_csv(diet_events_csv, index=False)
        del diet
        gc.collect()
    grouped_chunks = []
    for chunk in pd.read_csv(diet_events_csv, usecols=['participant_id', 'food_id', 'weight_g'], chunksize=500_000, low_memory=False):
        chunk['participant_id'] = chunk['participant_id'].astype(str)
        chunk['food_id'] = chunk['food_id'].astype(str)
        chunk['weight_g'] = pd.to_numeric(chunk['weight_g'], errors='coerce').fillna(0.0)
        grouped_chunks.append(chunk.groupby(['participant_id', 'food_id'], as_index=False)['weight_g'].sum())
    diet_slim = pd.concat(grouped_chunks, ignore_index=True).groupby(['participant_id', 'food_id'], as_index=False)['weight_g'].sum()
    diet_slim.to_csv(diet_slim_csv, index=False)
    print('Wrote slim diet table:', diet_slim_csv)
    return diet_slim_csv


def build_participant_x(fs, batch_size=120):
    fs_name = fs['name']
    existing = CVD_OUTPUTS / fs_name / f'X_{fs_name}_participant.csv'
    if existing.exists():
        print('Reusing X:', existing)
        return existing
    diet_slim_csv = ensure_diet_slim()
    print('Building X for', fs_name)
    ref = read_any(PROJECT_ROOT / fs['path'])
    ref_food_col = choose_ref_food_col(ref)
    cols = feature_columns(ref, ref_food_col, fs['feature_mode'])
    ref = ref[[ref_food_col] + cols].copy()
    ref['_food_join_id'] = ref[ref_food_col].astype(str)
    diet = pd.read_csv(diet_slim_csv, low_memory=False)
    diet['participant_id'] = diet['participant_id'].astype(str)
    diet['_food_join_id'] = diet['food_id'].astype(str)
    diet['weight_g'] = pd.to_numeric(diet['weight_g'], errors='coerce').fillna(0.0)
    total_grams = diet.groupby('participant_id')['weight_g'].sum().replace(0, np.nan)
    parts = []
    for start in range(0, len(cols), batch_size):
        batch = cols[start:start + batch_size]
        merged = diet[['participant_id', '_food_join_id', 'weight_g']].merge(ref[['_food_join_id'] + batch], on='_food_join_id', how='left')
        values = merged[batch].apply(pd.to_numeric, errors='coerce').fillna(0.0)
        if fs['feature_mode'] == 'enriched':
            scaled = values.mul(merged['weight_g'].to_numpy() / 100.0, axis=0)
            prefix = 'enriched_'
        elif fs['feature_mode'] in ['kg', 'embedding']:
            scaled = values.mul(merged['weight_g'].to_numpy(), axis=0)
            prefix = 'kg_' if fs['feature_mode'] == 'kg' else 'food_card_'
        else:
            raise ValueError(fs['feature_mode'])
        scaled['participant_id'] = merged['participant_id'].values
        agg = scaled.groupby('participant_id').sum(numeric_only=True)
        if fs['feature_mode'] in ['kg', 'embedding']:
            agg = agg.div(total_grams, axis=0).fillna(0.0)
        agg.columns = [prefix + str(c) for c in agg.columns]
        parts.append(agg)
        del merged, values, scaled, agg
        gc.collect()
    x = pd.concat(parts, axis=1).reset_index()
    fs_dir = CVD_OUTPUTS / fs_name
    fs_dir.mkdir(parents=True, exist_ok=True)
    out = fs_dir / f'X_{fs_name}_participant.csv'
    x.to_csv(out, index=False)
    print('Wrote X:', out)
    return out

x_paths = {fs['name']: build_participant_x(fs) for fs in FEATURE_SETS}
x_paths

## Discover Paper Figure Targets

The automatic target set mirrors the paper emphasis: body fat, waist/hip/BMI, folate and blood biomarkers, CGM traits, Nightingale biomarkers, and a derived 2-year overweight/obesity endpoint when longitudinal BMI is available.

In [ ]:
TARGET_TABLE_SPECS = [
    ('anthropometrics', 'anthropometrics'),
    ('body_composition', 'body_composition'),
    ('blood_tests', 'blood_tests'),
    ('cgm', 'cgm'),
    ('cgm', 'iglu'),
    ('cgm', 'iglu_daily'),
    ('nightingale_metabolomics', 'nightingale_metabolomics'),
]
COVARIATE_TABLE_SPECS = [
    ('anthropometrics', 'age_sex'),
    ('body_composition', 'age_sex'),
    ('blood_tests', 'age_sex'),
    ('cgm', 'age_sex'),
    ('nightingale_metabolomics', 'age_sex'),
    ('population', 'population'),
]
TARGET_REGEX = re.compile(
    r'body.*fat|body_comp.*fat|fat.*percent|fat_mass|visceral|vat|sat|'
    r'waist|hip|bmi|body_mass_index|folate|folic|b9|'
    r'triglyceride|cholesterol|hdl|ldl|glucose|glyca|hba1c|hemoglobin.*a1c|'
    r'alt|ast|ggt|creatinine|urate|cgm|time.*range|tir|mean.*glucose|average.*glucose|'
    r'gmi|j_index|mage|conga|modd|auc|cv|sd|obes|overweight',
    re.IGNORECASE,
)
TARGET_CATEGORY_RULES = [
    ('body_composition', re.compile(r'body_comp|body.*fat|fat.*percent|fat_mass|visceral|vat|sat', re.IGNORECASE)),
    ('anthropometry', re.compile(r'waist|hip|bmi|body_mass_index', re.IGNORECASE)),
    ('blood_biomarkers', re.compile(r'folate|folic|b9|triglyceride|cholesterol|hdl|ldl|glucose|glyca|hba1c|alt|ast|ggt|creatinine|urate', re.IGNORECASE)),
    ('cgm', re.compile(r'cgm|time.*range|tir|mean.*glucose|average.*glucose|gmi|j_index|mage|conga|modd|auc|cv|sd', re.IGNORECASE)),
    ('nightingale', re.compile(r'nightingale', re.IGNORECASE)),
    ('obesity_2y', re.compile(r'two_year|obes|overweight', re.IGNORECASE)),
]


def target_category(dataset, table, column):
    text = f'{dataset} {table} {column}'
    for category, pattern in TARGET_CATEGORY_RULES:
        if pattern.search(text):
            return category
    return 'other'

loaded_tables = {}
failed_table_specs = []
target_rows = []
dataset_briefs = []
for dataset, table in TARGET_TABLE_SPECS + COVARIATE_TABLE_SPECS:
    key = (dataset, table)
    if key in loaded_tables:
        continue
    frame, loader = try_load_pheno_table(dataset, table)
    if frame is None:
        failed_table_specs.append({'dataset': dataset, 'table': table})
        continue
    loaded_tables[key] = frame
    brief = dataframe_brief(dataset, table, frame)
    if brief:
        dataset_briefs.append(brief)

target_keys = set(TARGET_TABLE_SPECS)
for (dataset, table), frame in loaded_tables.items():
    if (dataset, table) not in target_keys:
        continue
    pid_col = find_participant_col(frame)
    if pid_col is None:
        continue
    for col in frame.columns:
        if col == pid_col:
            continue
        if TARGET_REGEX.search(str(col)):
            numeric = pd.api.types.is_numeric_dtype(frame[col])
            if not numeric and not any(token in str(col).lower() for token in ['obes', 'overweight']):
                continue
            nonnull = int(frame[col].notna().sum())
            if nonnull == 0:
                continue
            target_id = re.sub(r'[^A-Za-z0-9_]+', '_', f'{dataset}__{table}__{col}').strip('_')
            target_rows.append({
                'target_id': target_id,
                'category': target_category(dataset, table, col),
                'dataset': dataset,
                'table': table,
                'participant_col': pid_col,
                'column': col,
                'numeric': bool(numeric),
                'nonnull': nonnull,
                'nunique': int(frame[col].nunique(dropna=True)),
                'derived': False,
            })

dataset_briefs = pd.DataFrame(dataset_briefs).drop_duplicates()
target_catalog = pd.DataFrame(target_rows).drop_duplicates('target_id') if target_rows else pd.DataFrame()
failed_table_specs = pd.DataFrame(failed_table_specs)
dataset_briefs.to_csv(OUT_DIR / 'target_dataset_briefs.csv', index=False)
target_catalog.to_csv(OUT_DIR / 'paper_figure_target_catalog.csv', index=False)
failed_table_specs.to_csv(OUT_DIR / 'failed_target_table_loads.csv', index=False)

print('Loaded target/covariate tables:', len(loaded_tables))
print('Candidate targets:', len(target_catalog))
if not failed_table_specs.empty:
    print('Tables not available in this TRE release:')
    display(failed_table_specs)
display(dataset_briefs)
display(target_catalog.sort_values(['category', 'dataset', 'column']).head(120) if not target_catalog.empty else target_catalog)

In [ ]:
SELECTED_TARGET_IDS = []
PREFERRED_TARGET_TERMS = [
    'body_comp_total_region_percent_fat', 'total_region_percent_fat',
    'body_comp_trunk_region_percent_fat', 'trunk_region_percent_fat',
    'total_scan_vat_volume', 'total_scan_vat_mass', 'visceral',
    'waist_circumference', 'hip_circumference', 'bmi',
    'folate', 'folic', 'bt__glucose_float_value', 'bt__hba1c_float_value',
    'bt__triglycerides_float_value', 'bt__hdl_cholesterol_float_value', 'bt__ldl_cholesterol_float_value',
    'mean_glucose', 'average_glucose', 'cgm_mean', 'gmi', 'cgm_cv', 'time_in_range',
]


def target_priority(row):
    text = f"{row['dataset']} {row['table']} {row['column']}".lower()
    for i, term in enumerate(PREFERRED_TARGET_TERMS):
        if term.lower() in text:
            return i
    category_order = {
        'body_composition': 100,
        'anthropometry': 200,
        'blood_biomarkers': 300,
        'cgm': 400,
        'nightingale': 500,
        'obesity_2y': 600,
        'other': 999,
    }
    return category_order.get(row.get('category', 'other'), 999)

if target_catalog.empty:
    raise ValueError('No paper-figure targets discovered. Open target_dataset_briefs.csv and verify TRE PhenoLoader table names.')

if SELECTED_TARGET_IDS:
    selected_catalog = target_catalog[target_catalog['target_id'].isin(SELECTED_TARGET_IDS)].copy()
    missing = sorted(set(SELECTED_TARGET_IDS) - set(selected_catalog['target_id']))
    if missing:
        print('Requested target IDs not found:', missing)
else:
    selected_catalog = target_catalog.copy()
    selected_catalog['priority'] = selected_catalog.apply(target_priority, axis=1)
    selected_catalog = (
        selected_catalog
        .sort_values(['priority', 'nonnull'], ascending=[True, False])
        .groupby('category', group_keys=False)
        .head(8)
        .sort_values(['priority', 'nonnull'], ascending=[True, False])
        .head(MAX_TARGETS)
    )

selected_catalog.to_csv(OUT_DIR / 'selected_paper_figure_targets.csv', index=False)
print('Selected targets:', len(selected_catalog))
display(selected_catalog)

## Build Participant-Level Targets And Covariates

In [ ]:
def choose_participant_target_value(frame, pid_col, col):
    keep = [pid_col, col]
    date_cols = [c for c in ['collection_date', 'collection_timestamp', 'date', 'timestamp'] if c in frame.columns]
    part = frame[keep + date_cols].copy()
    part = part.rename(columns={pid_col: 'participant_id', col: 'target_value'})
    part['participant_id'] = normalize_pid_series(part['participant_id'])
    if not pd.api.types.is_numeric_dtype(part['target_value']):
        part['target_value'] = part['target_value'].astype(str).str.lower().isin(['1', 'true', 'yes', 'overweight', 'obese', 'obesity']).astype(float)
    else:
        part['target_value'] = pd.to_numeric(part['target_value'], errors='coerce')
    part = part.dropna(subset=['target_value'])
    if part.empty:
        return pd.DataFrame(columns=['participant_id', 'target_value'])
    if date_cols:
        date_col = date_cols[0]
        part[date_col] = pd.to_datetime(part[date_col], errors='coerce')
        part = part.sort_values(['participant_id', date_col])
        part = part.groupby('participant_id', as_index=False).first()
    else:
        part = part.groupby('participant_id', as_index=False)['target_value'].mean()
    return part[['participant_id', 'target_value']]


def find_first_matching_column(df, patterns):
    for pat in patterns:
        rx = re.compile(pat, re.IGNORECASE)
        exact = [c for c in df.columns if rx.fullmatch(str(c))]
        if exact:
            return exact[0]
        partial = [c for c in df.columns if rx.search(str(c))]
        if partial:
            return partial[0]
    return None


def build_two_year_overweight_target():
    frame = loaded_tables.get(('anthropometrics', 'anthropometrics'))
    if frame is None:
        frame, _ = try_load_pheno_table('anthropometrics', 'anthropometrics')
    if frame is None or frame.empty:
        return None, None
    pid_col = find_participant_col(frame)
    bmi_col = find_first_matching_column(frame, [r'bmi', r'body.*mass.*index'])
    date_col = find_first_matching_column(frame, [r'collection_timestamp', r'collection_date', r'date', r'timestamp'])
    if pid_col is None or bmi_col is None or date_col is None:
        print('Could not derive 2-year overweight/obesity target. Need participant, BMI, and date columns.')
        return None, None
    tmp = frame[[pid_col, bmi_col, date_col]].copy().rename(columns={pid_col: 'participant_id', bmi_col: 'bmi', date_col: 'date'})
    tmp['participant_id'] = normalize_pid_series(tmp['participant_id'])
    tmp['bmi'] = pd.to_numeric(tmp['bmi'], errors='coerce')
    tmp['date'] = pd.to_datetime(tmp['date'], errors='coerce')
    tmp = tmp.dropna(subset=['participant_id', 'bmi', 'date']).sort_values(['participant_id', 'date'])
    if tmp.empty:
        return None, None
    baseline = tmp.groupby('participant_id', as_index=False).first().rename(columns={'date': 'baseline_date', 'bmi': 'baseline_bmi'})
    merged = tmp.merge(baseline[['participant_id', 'baseline_date']], on='participant_id', how='inner')
    merged['days_from_baseline'] = (merged['date'] - merged['baseline_date']).dt.days
    follow = merged[(merged['days_from_baseline'] >= 540) & (merged['days_from_baseline'] <= 900)].copy()
    if follow.empty:
        print('No BMI follow-up rows found in the 18-30 month window.')
        return None, None
    follow['distance_to_2y'] = (follow['days_from_baseline'] - 730).abs()
    follow = follow.sort_values(['participant_id', 'distance_to_2y']).groupby('participant_id', as_index=False).first()
    out = follow[['participant_id', 'bmi', 'days_from_baseline']].copy()
    out['two_year_overweight_or_obesity_bmi25'] = (out['bmi'] >= 25.0).astype(int)
    meta = {
        'target_id': 'derived__two_year_overweight_or_obesity_bmi25',
        'category': 'obesity_2y',
        'dataset': 'anthropometrics',
        'table': 'anthropometrics',
        'participant_col': 'participant_id',
        'column': 'two_year_overweight_or_obesity_bmi25',
        'numeric': True,
        'nonnull': int(out['two_year_overweight_or_obesity_bmi25'].notna().sum()),
        'nunique': int(out['two_year_overweight_or_obesity_bmi25'].nunique(dropna=True)),
        'derived': True,
    }
    return out[['participant_id', 'two_year_overweight_or_obesity_bmi25']], meta

target_parts = []
for _, row in selected_catalog.iterrows():
    frame = loaded_tables[(row['dataset'], row['table'])]
    part = choose_participant_target_value(frame, row['participant_col'], row['column'])
    part = part.rename(columns={'target_value': row['target_id']})
    target_parts.append(part)

derived_2y, derived_2y_meta = build_two_year_overweight_target()
if derived_2y is not None and derived_2y_meta is not None:
    derived_2y = derived_2y.rename(columns={'two_year_overweight_or_obesity_bmi25': derived_2y_meta['target_id']})
    target_parts.append(derived_2y)
    selected_catalog = pd.concat([selected_catalog, pd.DataFrame([derived_2y_meta])], ignore_index=True)
    print('Added derived 2-year overweight/obesity target:', derived_2y_meta['nonnull'])

if not target_parts:
    raise ValueError('No target tables were built. Check selected_catalog.')

targets_wide = target_parts[0]
for part in target_parts[1:]:
    targets_wide = targets_wide.merge(part, on='participant_id', how='outer')

targets_path = OUT_DIR / 'paper_figure_targets_participant.csv'
selected_catalog.to_csv(OUT_DIR / 'selected_paper_figure_targets.csv', index=False)
targets_wide.to_csv(targets_path, index=False)
print('Wrote targets:', targets_path, targets_wide.shape)
display(targets_wide.head())

In [ ]:
COVARIATE_REGEX = re.compile(r'^age$|age_at|sex$|gender$|year_of_birth', re.IGNORECASE)
covariate_frames = []
for (dataset, table), frame in loaded_tables.items():
    if (dataset, table) not in set(COVARIATE_TABLE_SPECS):
        continue
    pid_col = find_participant_col(frame)
    if pid_col is None:
        continue
    matches = [c for c in frame.columns if COVARIATE_REGEX.search(str(c))]
    if not matches:
        continue
    part = frame[[pid_col] + matches].copy().rename(columns={pid_col: 'participant_id'})
    part['participant_id'] = normalize_pid_series(part['participant_id'])
    rename = {}
    for c in matches:
        lc = str(c).lower()
        if 'sex' in lc or 'gender' in lc:
            rename[c] = 'sex'
        elif 'year_of_birth' in lc:
            rename[c] = 'year_of_birth'
        elif 'age' in lc:
            rename[c] = 'age'
    part = part.rename(columns=rename)
    keep = ['participant_id'] + [c for c in ['age', 'sex', 'year_of_birth'] if c in part.columns]
    part = part[keep].copy()
    if 'age' in part.columns:
        part['age'] = pd.to_numeric(part['age'], errors='coerce')
    if 'year_of_birth' in part.columns and 'age' not in part.columns:
        part['year_of_birth'] = pd.to_numeric(part['year_of_birth'], errors='coerce')
        part['age'] = 2022 - part['year_of_birth']
        part = part.drop(columns=['year_of_birth'])
    elif 'year_of_birth' in part.columns:
        part = part.drop(columns=['year_of_birth'])
    part = part.groupby('participant_id', as_index=False).first()
    covariate_frames.append(part)

covariates = pd.DataFrame({'participant_id': targets_wide['participant_id'].astype(str).unique()})
for part in covariate_frames:
    for col in [c for c in part.columns if c != 'participant_id']:
        if col not in covariates.columns:
            covariates = covariates.merge(part[['participant_id', col]], on='participant_id', how='left')
        else:
            add = part[['participant_id', col]].rename(columns={col: f'{col}_new'})
            covariates = covariates.merge(add, on='participant_id', how='left')
            covariates[col] = covariates[col].combine_first(covariates[f'{col}_new'])
            covariates = covariates.drop(columns=[f'{col}_new'])

if 'sex' in covariates.columns:
    sex_text = covariates['sex'].astype(str).str.lower()
    covariates['sex_norm'] = np.select(
        [sex_text.str.startswith('m') | sex_text.isin(['1', 'male']), sex_text.str.startswith('f') | sex_text.isin(['0', '2', 'female'])],
        ['male', 'female'],
        default=np.nan,
    )
else:
    covariates['sex_norm'] = np.nan

covariates_path = OUT_DIR / 'paper_figure_covariates_participant.csv'
covariates.to_csv(covariates_path, index=False)
print('Wrote covariates:', covariates_path, covariates.shape)
print('Covariate non-null counts:')
display(covariates.notna().sum())
display(covariates.head())

## Build Comparison Arms

In [ ]:
def load_x(feature_set):
    x = pd.read_csv(x_paths[feature_set], low_memory=False)
    x['participant_id'] = x['participant_id'].astype(str)
    if 'time_window' in x.columns:
        x = x.drop(columns=['time_window'])
    return x

basic_x = load_x('basic_nutrimatch')
all_basic_feature_cols = [c for c in basic_x.columns if c != 'participant_id' and pd.api.types.is_numeric_dtype(basic_x[c])]

PAPER_BASIC_PATTERNS = {
    'energy': re.compile(r'(^|_)energy($|_)|calorie|kcal', re.IGNORECASE),
    'protein': re.compile(r'(^|_)protein($|_)', re.IGNORECASE),
    'total_lipid_fat': re.compile(r'total_lipid|lipid|total_fat|(^|_)fat($|_)', re.IGNORECASE),
    'carbohydrate_by_difference': re.compile(r'carbohydrate|(^|_)carb($|_)', re.IGNORECASE),
    'fiber_total_dietary': re.compile(r'fiber|fibre', re.IGNORECASE),
    'sodium_na': re.compile(r'sodium|(^|_)na($|_)', re.IGNORECASE),
    'water': re.compile(r'(^|_)water($|_)', re.IGNORECASE),
    'alcohol_ethyl': re.compile(r'alcohol|ethyl', re.IGNORECASE),
}

def paper_basic_nutrient_kind(col):
    name = clean_feature_name(col)
    if name in BASIC_NUTRIENT_KEYS:
        return name
    # Match common TRE/cache variants such as energy_kcal, protein_g, sodium_mg,
    # and original NutriMatch labels after the enriched_/kg_ prefix has been stripped.
    for kind, pattern in PAPER_BASIC_PATTERNS.items():
        if pattern.search(name):
            return kind
    return None

paper_basic_cols_by_kind = {}
for col in all_basic_feature_cols:
    kind = paper_basic_nutrient_kind(col)
    if kind and kind not in paper_basic_cols_by_kind:
        paper_basic_cols_by_kind[kind] = col

paper_basic_cols = list(paper_basic_cols_by_kind.values())
print('Paper-basic nutrient columns found:', paper_basic_cols)
print('Paper-basic nutrient kinds found:', sorted(paper_basic_cols_by_kind))
if not paper_basic_cols:
    print('First 60 available basic_nutrimatch columns:')
    print(all_basic_feature_cols[:60])
    raise ValueError('No paper-basic nutrient columns were found in basic_nutrimatch X.')

covariate_cols = [c for c in ['age', 'sex'] if c in covariates.columns and covariates[c].notna().any()]
print('Using covariates:', covariate_cols)

arms = [
    {'arm': 'age_sex_only', 'feature_set': 'covariates', 'label': ARM_LABELS['age_sex_only'], 'x': covariates[['participant_id'] + covariate_cols].copy()},
    {'arm': 'paper_basic_nutrients', 'feature_set': 'basic_nutrimatch', 'label': ARM_LABELS['paper_basic_nutrients'], 'x': basic_x[['participant_id'] + paper_basic_cols].merge(covariates[['participant_id'] + covariate_cols], on='participant_id', how='left')},
    {'arm': 'nutrimatch_all', 'feature_set': 'basic_nutrimatch', 'label': ARM_LABELS['nutrimatch_all'], 'x': basic_x.merge(covariates[['participant_id'] + covariate_cols], on='participant_id', how='left')},
]
for fs in FEATURE_SETS:
    if fs['name'] == 'basic_nutrimatch':
        continue
    x = load_x(fs['name']).merge(covariates[['participant_id'] + covariate_cols], on='participant_id', how='left')
    arms.append({'arm': fs['name'], 'feature_set': fs['name'], 'label': ARM_LABELS[fs['name']], 'x': x})

for arm in arms:
    print(arm['arm'], arm['x'].shape, arm['label'])

## Paper-Style Model Evaluation

The paper text describes the three model inputs and reports R2 for continuous phenotype prediction and AUROC for 2-year overweight/obesity prediction. This notebook uses the same old fold-by-fold scoring convention already used in the prior NutriMatch notebook. If LightGBM is installed and `DDE_MODEL=lightgbm`, the runner will use it; otherwise the default is sklearn `HistGradientBoosting`, which matched the earlier TRE notebook behavior.

In [ ]:
def make_onehot():
    try:
        return OneHotEncoder(handle_unknown='ignore', sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown='ignore', sparse=False)


def model_for(task_type: str):
    if MODEL_NAME == 'lightgbm':
        if not LIGHTGBM_AVAILABLE:
            print('LightGBM requested but unavailable; falling back to HistGradientBoosting.')
        else:
            if task_type == 'classification':
                return LGBMClassifier(n_estimators=300, learning_rate=0.03, random_state=RANDOM_STATE, verbose=-1)
            return LGBMRegressor(n_estimators=300, learning_rate=0.03, random_state=RANDOM_STATE, verbose=-1)
    if task_type == 'classification':
        return HistGradientBoostingClassifier(max_iter=300, learning_rate=0.03, random_state=RANDOM_STATE)
    return HistGradientBoostingRegressor(max_iter=300, learning_rate=0.03, random_state=RANDOM_STATE)


def infer_task_type(target_id, y):
    if str(target_id).startswith('derived__two_year'):
        return 'classification'
    y2 = y.dropna()
    if y2.nunique() <= 2:
        return 'classification'
    return 'regression'


def pipeline_for(X: pd.DataFrame, task_type: str) -> Pipeline:
    numeric_cols = X.select_dtypes(include=[np.number, bool]).columns.tolist()
    cat_cols = [c for c in X.columns if c not in numeric_cols]
    pre = ColumnTransformer(
        transformers=[
            ('num', Pipeline([('impute', SimpleImputer()), ('scale', StandardScaler())]), numeric_cols),
            ('cat', Pipeline([('impute', SimpleImputer(strategy='most_frequent')), ('onehot', make_onehot())]), cat_cols),
        ],
        remainder='drop',
    )
    return Pipeline([('pre', pre), ('model', model_for(task_type))])


def usable_cv(y: pd.Series, task_type: str, n_splits: int = N_SPLITS):
    if task_type == 'classification':
        counts = y.value_counts(dropna=True)
        if counts.empty or counts.min() < 2:
            return None
        k = min(n_splits, int(counts.min()))
        if k < 2:
            return None
        return StratifiedKFold(n_splits=k, shuffle=True, random_state=RANDOM_STATE)
    if len(y) < n_splits:
        return None
    return KFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)


def evaluate_arm(X, y, participants, *, task_type, n_splits=N_SPLITS):
    valid = y.notna()
    X = X.loc[valid].copy()
    y = y.loc[valid].copy()
    participants = participants.loc[valid].copy()
    if len(y) < MIN_N_PER_TARGET:
        return None, pd.DataFrame()
    class_count = None
    if task_type == 'classification':
        y = y.astype(int) if pd.api.types.is_numeric_dtype(y) else y.astype(str)
        class_count = int(pd.Series(y).nunique())
        if class_count < 2 or pd.Series(y).value_counts().min() < 2:
            return None, pd.DataFrame()
    cv = usable_cv(pd.Series(y), task_type, n_splits=n_splits)
    if cv is None:
        return None, pd.DataFrame()
    estimator = pipeline_for(X, task_type)
    split_iter = cv.split(X, y) if task_type == 'classification' else cv.split(X)
    rows = []
    oof_rows = []
    for fold, (train_idx, test_idx) in enumerate(split_iter, start=1):
        est = clone(estimator)
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
        est.fit(X_train, y_train)
        pred = est.predict(X_test)
        if task_type == 'classification':
            row = {
                'fold': fold,
                'accuracy': float(accuracy_score(y_test, pred)),
                'balanced_accuracy': float(balanced_accuracy_score(y_test, pred)),
                'f1_macro': float(f1_score(y_test, pred, average='macro', zero_division=0)),
                'auroc': np.nan,
                'auprc': np.nan,
            }
            score = None
            try:
                if hasattr(est, 'predict_proba'):
                    proba = est.predict_proba(X_test)
                    score = proba[:, -1] if proba.ndim == 2 else proba
                else:
                    score = est.decision_function(X_test)
                if class_count == 2:
                    positive = sorted(pd.Series(y_test).dropna().unique())[-1]
                    y_bin = (pd.Series(y_test).to_numpy() == positive).astype(int)
                    row['auroc'] = float(roc_auc_score(y_bin, score))
                    row['auprc'] = float(average_precision_score(y_bin, score))
                else:
                    row['auroc'] = float(roc_auc_score(y_test, score, multi_class='ovr'))
            except Exception:
                pass
            for pid, yt, yp, sc in zip(participants.iloc[test_idx], y_test, pred, score if score is not None else [np.nan] * len(y_test)):
                oof_rows.append({'participant_id': pid, 'fold': fold, 'y_true': yt, 'y_pred': yp, 'y_score': sc})
            rows.append(row)
        else:
            try:
                pr = float(pearsonr(y_test, pred).statistic)
            except Exception:
                pr = np.nan
            rows.append({
                'fold': fold,
                'r2': float(r2_score(y_test, pred)),
                'rmse': float(np.sqrt(mean_squared_error(y_test, pred))),
                'pearson_r': pr,
            })
            for pid, yt, yp in zip(participants.iloc[test_idx], y_test, pred):
                oof_rows.append({'participant_id': pid, 'fold': fold, 'y_true': yt, 'y_pred': yp, 'y_score': np.nan})
    fold_df = pd.DataFrame(rows)
    out = {
        'n': int(len(y)),
        'feature_count': int(X.shape[1]),
        'task': f'classification ({class_count} classes)' if task_type == 'classification' else 'regression',
        'task_type': task_type,
        'class_count': class_count,
        'model': MODEL_NAME if MODEL_NAME != 'lightgbm' or LIGHTGBM_AVAILABLE else 'hist_gradient_boosting',
    }
    for col in fold_df.columns:
        if col == 'fold':
            continue
        out[f'{col}_mean'] = float(fold_df[col].mean())
        out[f'{col}_std'] = float(fold_df[col].std())
    return out, pd.DataFrame(oof_rows)

In [ ]:
results_path = OUT_DIR / 'paper_figure_model_results.csv'
oof_path = OUT_DIR / 'paper_figure_oof_predictions.csv'

if RUN_TRAINING:
    result_rows = []
    oof_all = []
    stratum_values = ['all']
    if ENABLE_SEX_STRATA and 'sex_norm' in covariates.columns and covariates['sex_norm'].notna().any():
        stratum_values += ['female', 'male']
    for target_id in selected_catalog['target_id']:
        y_table = targets_wide[['participant_id', target_id]].copy()
        y_table['participant_id'] = y_table['participant_id'].astype(str)
        meta = selected_catalog[selected_catalog['target_id'].eq(target_id)].iloc[0].to_dict()
        task_type = infer_task_type(target_id, y_table[target_id])
        print('Target:', target_id, 'task:', task_type)
        for arm in arms:
            base = arm['x'].merge(y_table, on='participant_id', how='inner')
            base = base.merge(covariates[['participant_id', 'sex_norm']], on='participant_id', how='left')
            for stratum in stratum_values:
                if stratum == 'all':
                    merged = base.copy()
                else:
                    merged = base[base['sex_norm'].eq(stratum)].copy()
                if merged.empty:
                    continue
                participants = merged['participant_id'].copy()
                drop_cols = ['participant_id', target_id, 'sex_norm']
                X = merged.drop(columns=[c for c in drop_cols if c in merged.columns])
                y = merged[target_id]
                metrics, oof = evaluate_arm(X, y, participants, task_type=task_type, n_splits=N_SPLITS)
                if metrics is None:
                    print('  skipped', arm['arm'], stratum)
                    continue
                row = {
                    'target_id': target_id,
                    'target_column': meta['column'],
                    'target_dataset': meta['dataset'],
                    'target_table': meta['table'],
                    'target_group': meta['category'],
                    'stratum': stratum,
                    'arm': arm['arm'],
                    'feature_set': arm['feature_set'],
                    'label': arm['label'],
                    **metrics,
                }
                result_rows.append(row)
                if not oof.empty:
                    oof['target_id'] = target_id
                    oof['target_column'] = meta['column']
                    oof['target_group'] = meta['category']
                    oof['stratum'] = stratum
                    oof['arm'] = arm['arm']
                    oof['label'] = arm['label']
                    oof['task_type'] = task_type
                    oof_all.append(oof)
                key_metric = row.get('auroc_mean') if task_type == 'classification' else row.get('r2_mean')
                print(' ', arm['arm'], stratum, key_metric)
    results = pd.DataFrame(result_rows)
    oof_predictions = pd.concat(oof_all, ignore_index=True) if oof_all else pd.DataFrame()
    results.to_csv(results_path, index=False)
    oof_predictions.to_csv(oof_path, index=False)
    print('Wrote:', results_path, results.shape)
    print('Wrote:', oof_path, oof_predictions.shape)
elif results_path.exists():
    results = pd.read_csv(results_path, low_memory=False)
    oof_predictions = pd.read_csv(oof_path, low_memory=False) if oof_path.exists() else pd.DataFrame()
    print('Loaded saved results:', results_path, results.shape)
else:
    results = pd.DataFrame()
    oof_predictions = pd.DataFrame()
    print('Training skipped and no saved results found yet. Run the background runner first.')

display(results.head() if not results.empty else results)

## Metric Tables

In [ ]:
def ensure_metric_cols(df):
    df = df.copy()
    for old, new in {
        'r2_mean': 'r2', 'rmse_mean': 'rmse', 'pearson_r_mean': 'pearson_r',
        'auc_mean': 'auroc', 'auroc_mean': 'auroc', 'auprc_mean': 'auprc',
        'accuracy_mean': 'accuracy', 'balanced_accuracy_mean': 'balanced_accuracy', 'f1_macro_mean': 'f1_macro',
    }.items():
        if old in df.columns and new not in df.columns:
            df[new] = df[old]
    for col in ['r2', 'rmse', 'pearson_r', 'accuracy', 'balanced_accuracy', 'f1_macro', 'auroc', 'auprc']:
        if col not in df.columns:
            df[col] = np.nan
    if 'task_type' in df.columns:
        df['primary_metric_name'] = np.where(df['task_type'].eq('classification'), 'AUROC', 'R2')
        df['primary_metric'] = np.where(df['task_type'].eq('classification'), df['auroc'], df['r2'])
    return df

results = ensure_metric_cols(results)
if not results.empty:
    metric_table = results.copy()
    metric_table.to_csv(OUT_DIR / 'paper_figure_metric_table.csv', index=False)
    arm_summary = results.groupby(['arm', 'label', 'model', 'task_type', 'stratum'], as_index=False).agg(
        targets=('target_id', 'nunique'),
        mean_r2=('r2', 'mean'),
        median_r2=('r2', 'median'),
        mean_rmse=('rmse', 'mean'),
        mean_accuracy=('accuracy', 'mean'),
        mean_auroc=('auroc', 'mean'),
        mean_auprc=('auprc', 'mean'),
        mean_primary_metric=('primary_metric', 'mean'),
    )
    arm_summary.to_csv(OUT_DIR / 'paper_figure_arm_summary.csv', index=False)
    best_by_target = (
        metric_table.dropna(subset=['primary_metric'])
        .sort_values('primary_metric', ascending=False)
        .groupby(['target_id', 'task_type', 'stratum'], as_index=False)
        .head(1)
    )
    best_by_target.to_csv(OUT_DIR / 'paper_figure_best_by_target.csv', index=False)
    print('Wrote metric tables to:', OUT_DIR)
    display(metric_table.sort_values(['target_group', 'target_id', 'stratum', 'primary_metric'], ascending=[True, True, True, False]).head(120))
    display(arm_summary.sort_values(['task_type', 'stratum', 'mean_primary_metric'], ascending=[True, True, False]))
else:
    metric_table = pd.DataFrame()
    arm_summary = pd.DataFrame()
    best_by_target = pd.DataFrame()

## Figure 3a-Like Plot: Phenotype Prediction R2

In [ ]:
def save_fig(path):
    path.parent.mkdir(parents=True, exist_ok=True)
    plt.tight_layout()
    plt.savefig(path, dpi=220, bbox_inches='tight')
    print('Wrote:', path)

if not results.empty:
    reg = results[(results['task_type'].eq('regression')) & (results['stratum'].isin(['all', 'female', 'male']))].copy()
    reg['label'] = pd.Categorical(reg['label'], categories=[ARM_LABELS[a] for a in ARM_ORDER if a in ARM_LABELS], ordered=True)
    if not reg.empty:
        fig, axes = plt.subplots(1, len(sorted(reg['stratum'].unique())), figsize=(6 * len(sorted(reg['stratum'].unique())), 7), sharex=True, sharey=True)
        if not isinstance(axes, np.ndarray):
            axes = np.array([axes])
        for ax, stratum in zip(axes, sorted(reg['stratum'].unique())):
            p = reg[reg['stratum'].eq(stratum)]
            summary = p.groupby(['target_group', 'label'], observed=False, as_index=False)['r2'].median()
            sns.barplot(data=summary, y='target_group', x='r2', hue='label', ax=ax, orient='h')
            ax.axvline(0, color='black', lw=0.8)
            ax.set_title(f'Figure 3a-like R2, {stratum}')
            ax.set_xlabel('Median fold R2')
            ax.set_ylabel('Phenotype group')
            if ax is not axes[-1]:
                ax.legend_.remove()
        path = FIG_DIR / 'fig3a_like_r2_by_phenotype_group_and_stratum.png'
        save_fig(path)
        plt.show()

        best_examples = (
            reg[~reg['arm'].isin(PAPER_ARMS)]
            .merge(reg[reg['arm'].eq('nutrimatch_all')][['target_id', 'stratum', 'r2']].rename(columns={'r2': 'nutrimatch_all_r2'}), on=['target_id', 'stratum'], how='left')
        )
        best_examples['delta_r2_vs_nutrimatch_all'] = best_examples['r2'] - best_examples['nutrimatch_all_r2']
        best_examples = best_examples.sort_values('delta_r2_vs_nutrimatch_all', ascending=False).head(30)
        best_examples.to_csv(OUT_DIR / 'fig3a_best_enhanced_examples_vs_nutrimatch_all.csv', index=False)
        if not best_examples.empty:
            plt.figure(figsize=(10, 8))
            sns.barplot(data=best_examples, y='target_column', x='delta_r2_vs_nutrimatch_all', hue='label')
            plt.axvline(0, color='black', lw=0.8)
            plt.title('Best enhanced examples vs NutriMatch all nutrients')
            plt.xlabel('Delta R2')
            plt.ylabel('Target')
            save_fig(FIG_DIR / 'fig3a_best_enhanced_examples_vs_nutrimatch_all.png')
            plt.show()
    else:
        print('No regression rows available for Figure 3a-like plot.')
else:
    print('No results loaded.')

## Figure 3b-Like Plot: Nightingale Correlations

This is a fast correlation analogue of the paper heatmap. It computes participant-level correlations between Nightingale biomarkers and dietary features. Values below absolute 0.1 are masked in the heatmap.

In [ ]:
def compute_corr_heatmap_for_arm(arm, max_features=80, max_biomarkers=40):
    night = loaded_tables.get(('nightingale_metabolomics', 'nightingale_metabolomics'))
    if night is None:
        night, _ = try_load_pheno_table('nightingale_metabolomics', 'nightingale_metabolomics')
    if night is None or night.empty:
        return pd.DataFrame()
    pid_col = find_participant_col(night)
    if pid_col is None:
        return pd.DataFrame()
    biomarker_cols = [c for c in night.select_dtypes(include=[np.number]).columns if c != pid_col]
    biomarker_cols = sorted(biomarker_cols, key=lambda c: night[c].notna().sum(), reverse=True)[:max_biomarkers]
    y = night[[pid_col] + biomarker_cols].rename(columns={pid_col: 'participant_id'}).copy()
    y['participant_id'] = y['participant_id'].astype(str)
    y = y.groupby('participant_id', as_index=False).mean(numeric_only=True)
    x = arm['x'].copy()
    feature_cols = [c for c in x.select_dtypes(include=[np.number]).columns if c != 'participant_id']
    feature_cols = sorted(feature_cols, key=lambda c: x[c].var(skipna=True), reverse=True)[:max_features]
    x = x[['participant_id'] + feature_cols]
    merged = x.merge(y, on='participant_id', how='inner')
    if len(merged) < MIN_N_PER_TARGET:
        return pd.DataFrame()
    corr = merged[feature_cols + biomarker_cols].corr(method='spearman').loc[feature_cols, biomarker_cols]
    return corr

if not results.empty:
    candidate_arms = [a for a in arms if a['arm'] in ['nutrimatch_all', 'denovo_cardiometabolic', 'nutrimatch_cardiometabolic', 'denovo_microbiome', 'nutrimatch_microbiome']]
    corr_summaries = []
    corr_by_arm = {}
    for arm in candidate_arms:
        corr = compute_corr_heatmap_for_arm(arm)
        if corr.empty:
            continue
        corr_by_arm[arm['arm']] = corr
        corr.to_csv(OUT_DIR / f'fig3b_like_nightingale_correlations_{arm["arm"]}.csv')
        corr_summaries.append({'arm': arm['arm'], 'label': arm['label'], 'mean_abs_top_corr': corr.abs().max(axis=1).mean()})
    corr_summary = pd.DataFrame(corr_summaries).sort_values('mean_abs_top_corr', ascending=False) if corr_summaries else pd.DataFrame()
    corr_summary.to_csv(OUT_DIR / 'fig3b_like_correlation_arm_summary.csv', index=False)
    display(corr_summary)
    for arm_name in [x for x in ['nutrimatch_all'] if x in corr_by_arm] + ([corr_summary.iloc[0]['arm']] if not corr_summary.empty and corr_summary.iloc[0]['arm'] != 'nutrimatch_all' else []):
        corr = corr_by_arm[arm_name]
        row_order = corr.abs().max(axis=1).sort_values(ascending=False).head(25).index
        col_order = corr.abs().max(axis=0).sort_values(ascending=False).head(25).index
        plot_corr = corr.loc[row_order, col_order].mask(corr.loc[row_order, col_order].abs() < 0.1)
        plt.figure(figsize=(11, 9))
        sns.heatmap(plot_corr, cmap='vlag', center=0, vmin=-1, vmax=1)
        plt.title(f'Figure 3b-like Nightingale correlations: {ARM_LABELS.get(arm_name, arm_name)}')
        plt.xlabel('Nightingale biomarker')
        plt.ylabel('Diet feature')
        save_fig(FIG_DIR / f'fig3b_like_nightingale_correlations_{arm_name}.png')
        plt.show()
else:
    print('No results loaded; skipping correlation heatmap.')

## Figure 3c-Like Plot: 2-Year Overweight/Obesity ROC

In [ ]:
if not oof_predictions.empty:
    roc_data = oof_predictions[
        oof_predictions['target_id'].eq('derived__two_year_overweight_or_obesity_bmi25')
        & oof_predictions['stratum'].eq('all')
    ].copy()
    if not roc_data.empty:
        roc_rows = []
        plt.figure(figsize=(7, 6))
        for arm in [a for a in ARM_ORDER if a in set(roc_data['arm'])]:
            p = roc_data[roc_data['arm'].eq(arm)].dropna(subset=['y_true', 'y_score'])
            if p.empty or p['y_true'].nunique() < 2:
                continue
            y_true = p['y_true'].astype(int)
            y_score = pd.to_numeric(p['y_score'], errors='coerce')
            keep = y_score.notna()
            y_true = y_true.loc[keep]
            y_score = y_score.loc[keep]
            if y_true.nunique() < 2:
                continue
            fpr, tpr, _ = roc_curve(y_true, y_score)
            auc = roc_auc_score(y_true, y_score)
            roc_rows.append({'arm': arm, 'label': ARM_LABELS.get(arm, arm), 'auroc': auc, 'n': len(y_true)})
            plt.plot(fpr, tpr, lw=2, label=f'{ARM_LABELS.get(arm, arm)} (AUC={auc:.3f})')
        plt.plot([0, 1], [0, 1], '--', color='black', lw=1)
        plt.xlabel('False positive rate')
        plt.ylabel('True positive rate')
        plt.title('Figure 3c-like 2-year overweight/obesity ROC')
        plt.legend(fontsize=8)
        save_fig(FIG_DIR / 'fig3c_like_two_year_overweight_obesity_roc.png')
        plt.show()
        roc_metrics = pd.DataFrame(roc_rows).sort_values('auroc', ascending=False)
        roc_metrics.to_csv(OUT_DIR / 'fig3c_two_year_overweight_obesity_roc_metrics.csv', index=False)
        display(roc_metrics)
    else:
        print('No derived 2-year overweight/obesity OOF predictions found.')
else:
    print('No OOF predictions loaded.')

## Paper-Ready Tables

In [ ]:
if not results.empty:
    display_cols = [
        'target_group', 'target_column', 'stratum', 'task', 'label', 'model', 'n', 'feature_count',
        'r2', 'rmse', 'pearson_r', 'accuracy', 'balanced_accuracy', 'auroc', 'auprc', 'primary_metric',
    ]
    display_cols = [c for c in display_cols if c in metric_table.columns]
    paper_table = metric_table[display_cols].sort_values(['target_group', 'target_column', 'stratum', 'primary_metric'], ascending=[True, True, True, False])
    paper_table.to_csv(OUT_DIR / 'paper_ready_metric_table.csv', index=False)
    print('Wrote:', OUT_DIR / 'paper_ready_metric_table.csv')
    display(paper_table.head(200))
else:
    print('No saved result table yet.')